# Tahap 04.1: Feature Extraction & Selection

Tahap ini mengubah data teks yang telah dibersihkan menjadi representasi numerik menggunakan metode *Term Frequency-Inverse Document Frequency* (TF-IDF). Setelah ekstraksi fitur selesai, tahap dilanjutkan dengan *Feature Selection* menggunakan metode Chi-Square ($\chi^2$) untuk mereduksi dimensi matriks dengan menyeleksi atribut (kata) yang paling berpengaruh terhadap label kelas.

In [ ]:
# --- LIBRARIES ---
import os
import pickle
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# --- 1. KONFIGURASI DIREKTORI & PATH ---
BASE_DIR = r'C:\Users\Asus\Desktop\SKRIPSI\1. MAIN'
TRAIN_FILE = os.path.join(BASE_DIR, 'data_train_final_2.0.csv')
TEST_FILE = os.path.join(BASE_DIR, 'data_test_final_2.0.csv')

# --- 2. FUNGSI EKSTRAKSI FITUR (TF-IDF) ---
def execute_tfidf():
    """
    Melakukan proses pemuatan data latih dan uji, inisialisasi model TF-IDF,
    serta transformasi teks menjadi matriks numerik.
    """
    # Memuat data hasil splitting
    train_df = pd.read_csv(TRAIN_FILE)
    test_df = pd.read_csv(TEST_FILE)

    # Memastikan tidak ada nilai NaN di kolom teks
    X_train = train_df['text'].fillna('')
    X_test = test_df['text'].fillna('')

    # Inisialisasi TF-IDF Vectorizer (Unigram)
    tfidf = TfidfVectorizer(ngram_range=(1, 1))

    # Proses Fit (pembelajaran kosa kata) & Transform (konversi matriks)
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    # --- LAPORAN HASIL EKSTRAKSI ---
    print("="*60)
    print("LAPORAN FEATURE EXTRACTION (TF-IDF)")
    print("="*60)
    print(f"Jumlah Baris (Training) : {X_train_tfidf.shape[0]}")
    print(f"Jumlah Fitur/Kata Unik  : {X_train_tfidf.shape[1]}")
    print("-" * 60)
    
    # Penyimpanan Model TF-IDF Dasar
    with open(os.path.join(BASE_DIR, 'tfidf_model_2.0.pkl'), 'wb') as f:
        pickle.dump(tfidf, f)
        
    print("Model TF-IDF utama telah berhasil disimpan.")
    
    return X_train_tfidf, X_test_tfidf, tfidf

# --- 3. EKSEKUSI UTAMA & PENYIMPANAN DATA EKSPERIMEN ---
if __name__ == "__main__":
    # Eksekusi fungsi TF-IDF
    X_train_tfidf, X_test_tfidf, tfidf = execute_tfidf()
    
    # --- BLOK REVISI: Penyimpanan Data Murni untuk Eksperimen Tambahan ---
    print("\n[INFO REVISI] Menyiapkan data murni untuk eksperimen Tanpa Chi-Square...")
    
    df_train_temp = pd.read_csv(TRAIN_FILE)
    df_test_temp = pd.read_csv(TEST_FILE)
    
    y_train_full = df_train_temp['label_pks']
    y_test_full = df_test_temp['label_pks']
    
    # Simpan fitur murni dan label ke dalam satu file Pickle
    save_path = os.path.join(BASE_DIR, 'tfidf_model_rev_2.0.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump((X_train_tfidf, X_test_tfidf, y_train_full, y_test_full), f)
        
    print(f"BERHASIL! Matriks TF-IDF murni tersimpan di: {save_path}")

## 4.1 Seleksi Fitur (Chi-Square)
Pemilihan sejumlah fitur terbaik ($K$) berdasarkan nilai statistik Chi-Square tertinggi antara fitur dan target kelas. Proses ini krusial untuk mencegah *overfitting* dan mempercepat proses komputasi pada saat pelatihan model.

In [ ]:
# --- LIBRARIES TAMBAHAN ---
from scipy import sparse
from sklearn.feature_selection import SelectKBest, chi2

# --- 1. PEMUATAN LABEL KELAS ---
train_df = pd.read_csv(os.path.join(BASE_DIR, 'data_train_final_2.0.csv'))
test_df = pd.read_csv(os.path.join(BASE_DIR, 'data_test_final_2.0.csv'))

y_train = train_df['label_pks'] 
y_test = test_df['label_pks']   

print("="*60)
print("MEMULAI PROSES FEATURE SELECTION (CHI-SQUARE)")
print("="*60)

# --- 2. KONFIGURASI DAN EKSEKUSI SELEKSI FITUR ---
# Menetapkan jumlah fitur optimal yang akan dipertahankan (K)
k_features = 1000 
selector = SelectKBest(score_func=chi2, k=k_features)

# Proses penyeleksian fitur pada matriks TF-IDF
X_train_selected = selector.fit_transform(X_train_tfidf, y_train)
X_test_selected = selector.transform(X_test_tfidf)

# --- 3. ANALISIS SKOR FITUR (Untuk Laporan Bab 4) ---
feature_names = tfidf.get_feature_names_out()

# Menyusun DataFrame hasil skor Chi-Square untuk setiap kata
df_scores = pd.DataFrame({
    'Keyword': feature_names,
    'Score': selector.scores_
}).sort_values(by='Score', ascending=False)

print(f"Berhasil menyaring dari {len(feature_names)} kata menjadi {k_features} kata terbaik.")
print("-" * 60)
print("10 KATA DENGAN SKOR CHI-SQUARE TERTINGGI:")
print(df_scores.head(10).to_string(index=False))
print("-" * 60)

# --- 4. PENYIMPANAN HASIL SELEKSI (ORIGINAL) ---
sparse.save_npz(os.path.join(BASE_DIR, 'X_train_selected_2.0.npz'), X_train_selected)
sparse.save_npz(os.path.join(BASE_DIR, 'X_test_selected_2.0.npz'), X_test_selected)

with open(os.path.join(BASE_DIR, 'selector_chi2_2.0.pkl'), 'wb') as f:
    pickle.dump(selector, f)

df_scores.to_csv(os.path.join(BASE_DIR, 'hasil_skor_chi2_lengkap_2.0.csv'), index=False)
print("Hasil seleksi matriks dan tabel skor Chi-Square telah disimpan.")

# --- 5. BLOK REVISI: Penyimpanan Data Lengkap untuk Baseline Model ---
print("\n[INFO REVISI] Menyimpan paket data Chi-Square lengkap (Fitur + Label)...")
save_path_rev = os.path.join(BASE_DIR, 'selector_chi2_rev_2.0.pkl')

with open(save_path_rev, 'wb') as f:
    pickle.dump((X_train_selected, X_test_selected, y_train, y_test), f)

print(f"BERHASIL! Paket data revisi tersimpan di: {save_path_rev}")
print("="*60)